# IAC Flow Matching — tek koşu eğitim notebook’u

**Elindeki üç sınıflı nnU-Net’ten devam eder.** Erken encoder donuk, son encoder katmanı ve decoder eğitilebilir. SDF çıktı kafası eski logit farklarıyla başlar. İki kanallı SDF maskesi flow integrasyonunun son durumundan üretilir.

**Bu notebook neyi yanıtlayacak?** “Segmentasyon öğrendi mi?” ve “ek akış adımları bağlantılılığa katkı sağladı mı?” ayrı ölçülür. Çok adımın iyi sonuç vermesi garanti değildir. Aynı tek eğitilmiş ağı NFE=1 ve NFE=4’te çalıştırmak yeniden eğitim değildir.

**Gerekli dosyalar:** kendi bg/L/R checkpoint’in, ona ait `plans.json`, `dataset.json`, `splits_final.json` ve zaten hazırlanmış nnU-Net preprocessed verisi. Hocanın binary/SDF aux script’i bu checkpoint’in yerine kullanılmaz.

Gerçek dental verin ve checkpoint’in burada bulunmadığı için GPU/ToothFairy başarısı henüz test edilmedi. `DEMO=True` seçeneği küçük sentetik veriyle mühendislik akışını çalıştırır; başarı sayıları üretmek için kullanılamaz. Orijinal veri ve checkpoint değişmez.

**Sıra:** bağımlılıklar → yollar/fold → CPU cache → özgün referans cache → gradyan/profil → tek eğitim/resume → sonuç ekranı. İlk cache işlemlerini GPU kiralamadan önce yapabilirsin.

## 1. Kod paketi ve ortam

Kod bu notebook’un içine gömülüdür; ZIP indirmek zorunlu değildir. Açılan `iacflow/*.py` dosyaları okunabilir. Aynı adlı değiştirilmiş kod dosyası otomatik ezilmez. Aşağıdaki paket hücresi uzun olduğu için kapalı görüntülenir.

In [ ]:
import base64, hashlib, io, os, sys, zipfile
from pathlib import Path

_PAYLOAD = "..."
_DIGEST = "ceeab4b27a0bf510fc8e174871658700b41933a30ed55e1bd06099f063bb1c42"
_raw = base64.b64decode(_PAYLOAD)
assert hashlib.sha256(_raw).hexdigest() == _DIGEST
CODE_ROOT = Path.cwd() / ("iac_flow_code_" + _DIGEST[:10])
with zipfile.ZipFile(io.BytesIO(_raw)) as _zip:
    for _item in _zip.infolist():
        _out = CODE_ROOT / _item.filename
        _data = _zip.read(_item.filename)
        if _out.exists() and _out.read_bytes() != _data:
            raise RuntimeError(f"Düzenlenmiş kod korunuyor: {_out}. Yeni bir çalışma klasörü aç.")
        _out.parent.mkdir(parents=True, exist_ok=True)
        if not _out.exists(): _out.write_bytes(_data)
sys.path.insert(0, str(CODE_ROOT))
print("Kod klasörü:", CODE_ROOT)
del _PAYLOAD, _raw

In [ ]:
# Mevcut PyTorch/CUDA otomatik yükseltilmez. Yalnızca eksik küçük bağımlılıklar kurulur.
INSTALL_MISSING = True
import importlib.util, importlib.metadata, subprocess
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Önce GPU sağlayıcının CUDA uyumlu PyTorch imajını seç. Bu hücre torch kurmaz.")
_deps = {
    "numpy":"numpy>=1.26", "scipy":"scipy>=1.11", "skimage":"scikit-image>=0.23",
    "pandas":"pandas>=2.1", "matplotlib":"matplotlib>=3.8",
}
_missing = [p for m,p in _deps.items() if importlib.util.find_spec(m) is None]
if _missing and INSTALL_MISSING:
    subprocess.check_call([sys.executable,"-m","pip","install",*_missing])
elif _missing:
    raise RuntimeError(f"Eksik paketler: {_missing}")
if importlib.util.find_spec("dynamic_network_architectures") is None:
    if not INSTALL_MISSING: raise RuntimeError("dynamic-network-architectures eksik")
    subprocess.check_call([sys.executable,"-m","pip","install","--no-deps","dynamic-network-architectures==0.4.4"])
# Sadece kaynak verin .b2nd ise ve blosc2 eksikse ayrıca: pip install 'blosc2>=3.0'
import torch, numpy as np, pandas as pd
from iacflow.core import IACFlow, load_backbone, validate_patch, check_head_initialization, atomic_json
from iacflow.data import load_splits, prepare_cpu_cache, cache_ready, PatchDataset
from iacflow.inference import prepare_references
from iacflow.train import default_config, validate_config, profile_training, train, evaluate, final_report, seed_all
try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = str
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok — yalnız CPU/demo")

## 2. Kendi dosyalarını bağla

`/EDIT/...` yollarını doldur. `preprocessed_data_identifier`, plans içindeki configuration ile aynı olmalı. Preprocessed klasörü normalize edilmiş/resample edilmiş veri içerir; burada yeni resampling yapılmaz. L/R isimlerini `dataset.json` üzerinden kontrol et.

`checkpoint_split_verified=True`, checkpoint’in validation hastalarını eğitimde görmediğini **kontrol ettikten sonra** konur. Yalnızca `fold_0` dosya adı bunu kanıtlamaz. Checkpoint tüm veriyle eğitilmişse o hastalar üzerinde bağımsız validation sonucu iddia edemeyiz.

Başlangıç kararı görünür: `eta=.01`, ±3 mm SDF, orijinal tahminden sınır ağırlığı, son encoder katmanı açık, **tek FM kaybı**. Herhangi bir otomatik mimari/çözünürlük değişikliği yok. `max_steps=15000` zorunlu eğitim süresi değildir; profil ve ara ölçümlerle bütçe kontrol edilir.

In [ ]:
DEMO = os.environ.get("IAC_FLOW_DEMO", "0") == "1"  # Gerçek kullanımda False; sentetik kontrol için True.
CPU_CACHE_ONLY = False  # GPU kiralamadan cache hazırlamak için True; GPU hücreleri atlanır.
CONFIG = default_config()
CONFIG.update(
    checkpoint_path="/EDIT/nnUNet_results/.../fold_0/checkpoint_final.pth",
    plans_path="/EDIT/nnUNet_results/.../plans.json",
    dataset_json_path="/EDIT/nnUNet_results/.../dataset.json",
    splits_path="/EDIT/nnUNet_preprocessed/Dataset504_IAC_LR/splits_final.json",
    preprocessed_dir="/EDIT/nnUNet_preprocessed/Dataset504_IAC_LR/nnUNetPlans_3d_fullres",
    preprocessed_data_identifier="nnUNetPlans_3d_fullres",
    cache_dir="/EDIT/local_ssd/iac_flow_cache",
    run_dir="/EDIT/local_ssd/iac_flow_run",
    backup_dir=None,  # Örnek: kalıcı disk üzerindeki /workspace/persistent/iac_flow_backup
    reference_dir=None,  # Yalnız manifest ile aynı checkpoint/grid/protokolü kanıtlanan .npy maskeler.
    configuration="3d_fullres", fold=0, left_id=1, right_id=2,
    checkpoint_split_verified=False,
    batch_size=1, accumulation=2, patch_size=None, # None = checkpoint'in planlanan patch'i
    max_steps=15000, max_session_hours=6.,
    num_workers=2, cache_workers=1, cpu_threads=2,
    first_probe_step=250, probe_every=1000, sentinel_cases=5,
    nfe_values=[1,4], primary_nfe=4,
    stop_on_no_flow_signal=True,
)
if DEMO:
    from iacflow.demo import create_demo_workspace
    CONFIG = create_demo_workspace(Path.cwd()/"iac_flow_synthetic_demo")
torch.set_num_threads(CONFIG["cpu_threads"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() and not CPU_CACHE_ONLY and not DEMO else "cpu")
seed_all(CONFIG["seed"],CONFIG["deterministic"])
display(pd.DataFrame({"ayar":list(CONFIG),"değer":[str(v) for v in CONFIG.values()]}))

In [ ]:
for _key in ("checkpoint_path","plans_path","dataset_json_path","splits_path"):
    if not Path(CONFIG[_key]).is_file(): raise FileNotFoundError(f"{_key}: {CONFIG[_key]}")
BACKBONE, INFO = load_backbone(CONFIG["checkpoint_path"],CONFIG["plans_path"],
                             CONFIG["dataset_json_path"],CONFIG["configuration"],CONFIG["fold"])
SPLITS = load_splits(CONFIG["splits_path"],CONFIG["fold"],CONFIG.get("patient_map"))
CONFIG = validate_config(CONFIG,INFO,SPLITS,DEVICE)
MODEL = IACFlow(BACKBONE,CONFIG["left_id"],CONFIG["right_id"],CONFIG["margin_scale"],CONFIG["adapter_hidden"])
validate_patch(MODEL,CONFIG["patch_size"])
print("Checkpoint bilgisi:", INFO)
print("Train / validation vaka:",len(SPLITS["train"]),len(SPLITS["val"]))
print("Parametre sözleşmesi:", MODEL.trainable_report())
if not INFO["fold_metadata_present"]:
    print("Checkpoint içinde fold metadata yok; doğruluk senin split doğrulamana bağlı.")

## 3. CPU ön hazırlık — sadece bir kez

GT → fiziksel SDF → ±3 mm → [−1,1] → maske. **Round-trip bütün geçerli voksellerde tam eşit değilse eğitim başlamaz.** EDT patch içinde hesaplanmaz; yapay anatomik uç oluşturulmaz. Uzak sabit alanı atlayan hesap tam hacim EDT’siyle aynı kırpılmış hedefi verir.

Cache `.npy` kullanır; eğitimde `mmap` ile yalnız gereken patch okunur. Yaklaşık 14 byte/voksel nihai cache için boş yer gerekir; geçici RAM ve orijinal dosyalar ayrıca yer kaplar. Cache worker sayısını CPU çekirdek sayısına körlemesine eşitleme: her worker bir hacim işler, RAM artar. Varsayılan 1.

In [ ]:
import shutil, time
print("Cache diski boş GiB:", round(shutil.disk_usage(Path(CONFIG["cache_dir"]).parent
      if Path(CONFIG["cache_dir"]).parent.exists() else Path.cwd()).free/2**30,1))
_start = time.perf_counter()
MANIFEST = prepare_cpu_cache(CONFIG,INFO,SPLITS)
print("CPU cache toplam dakika:", round((time.perf_counter()-_start)/60,2))
if CPU_CACHE_ONLY:
    print("CPU hazırlığı tamamlandı. GPU oturumunda aynı cache yolunu bağlayıp CPU_CACHE_ONLY=False seç.")

## 4. Aynı checkpoint’in referans maskeleri ve sınır ağırlıkları

Uyumlu önbellek varsa bu aşama çıkarım yapmaz. Yoksa orijinal nnU-Net bir defa her vaka için çalışır; süre/ETA görünür. Bu ek bir eğitim değildir, fakat ücretsiz de değildir. Eğitim minibatch’lerinde eski modele ikinci bir ileri geçiş yoktur. GT bu referans çıkarımının girdisi değildir.

Baseline ve Flow aynı preprocessed gridde, aynı patch/overlap ile, **TTA olmadan** ölçülür. Eski 0.905050 sonucu başka çıkarım/postprocess protokolüyle üretildiyse burada birebir tekrarlanmayabilir; eski ölçüm yanlış sayılmaz. Native NIfTI maskesinin şeklinin uyması doğru gridde olduğu anlamına gelmez.

In [ ]:
if not CPU_CACHE_ONLY:
    if DEVICE.type != "cuda" and not DEMO:
        raise RuntimeError("Gerçek eğitim için GPU görünmüyor. CPU cache hazır; GPU runtime seç.")
    MODEL = MODEL.to(DEVICE)
    # Orijinal checkpoint bu noktada henüz güncellenmedi.
    prepare_references(CONFIG,INFO,SPLITS,MODEL)
    CACHE_ID = cache_ready(CONFIG,INFO,SPLITS)
    print("Cache hazır:",CACHE_ID[:16])

## 5. Uzun koşudan önce gradyan ve hız kontrolü

Erken encoder’a gradyan gitmemeli; son encoder ve adaptörün son projeksiyonuna gitmeli. Sıfır adaptör ile temiz kafanın kararı orijinal logit kararına eşit olmalı. Profil gerçek forward/backward çalıştırır, optimizer güncellemesi yapmaz; RNG/buffer durumunu geri koyar.

Profilde AdamW moment belleği ve uzun doğrulama maliyeti yoktur. İlk gerçek güncellemelerde ek bellek gerekir; en az %20 boş VRAM bırak. OOM olursa notebook çözünürlüğü veya patch’i otomatik değiştirmez. `batch_size=1` ve accumulation bellek bütçesi için seçilidir; accumulation tek patch belleğini azaltmaz.

Öğrenme hedefi sözle: **“Görüntüye ve gürültülü ara geometriye bak, temiz kanal mesafesini kestir.”** Tahminin analitik hız dönüşümü FM hedefini sağlar. State ve zamanı kullanmadan görüntüden kestirmek de kaybın geçerli çözümüdür; bu yüzden ileride gerçek NFE katkısı ayrıca ölçülür.

In [ ]:
if not CPU_CACHE_ONLY:
    _ds = PatchDataset(CONFIG["cache_dir"],SPLITS["train"],CONFIG["patch_size"],samples=1,seed=CONFIG["seed"])
    _example = _ds[0]["image"][None].to(DEVICE)
    print(check_head_initialization(MODEL,_example))
    del _example,_ds
    PROFILE = profile_training(MODEL,CONFIG,SPLITS,repeats=2 if DEMO else 3)
    _env={"torch":torch.__version__,"cuda":torch.version.cuda,"device":str(DEVICE),"code_payload_sha256":_DIGEST}
    for _p in ("numpy","scipy","scikit-image","dynamic-network-architectures"):
        _env[_p]=importlib.metadata.version(_p)
    atomic_json(Path(CONFIG["run_dir"])/"environment.json",_env)
    atomic_json(Path(CONFIG["run_dir"])/"profile.json",PROFILE)

## 6. Tek eğitim koşusu / kesintiden devam

Bu hücre otomatik yeni deneyler başlatmaz. `latest.pt` varsa aynı koşuya devam eder. Başlangıç checkpoint’i, veri/ayar/code sözleşmesi, model, AdamW, AMP scaler, RNG ve sıradaki patch indeksi kontrol edilir. Yeni GPU oturumunda hücreleri sırayla tekrar çalıştırabilirsin; cache tekrar üretilmez.

İlk panel adım 250’de, sonra her 1000 adımda sabit en çok 5 validation vakası üzerinde NFE=1 ve NFE=4 ölçer. Bu seyrek de olsa pahalı olabilir; ilk panelin süresi logda görünür. Her panelde yalnız bir patch için temiz kafa state shuffle duyarlılığı da ölçülür. Panel içindeki NFE=4, NFE=1’den kalan görüntü özellik cache’inden yararlanabilir; süreleri bağımsız dağıtım latency kıyası sayma. Gerçek decoder/encoder çağrıları ayrı loglanır.

**Bütçe durdurması:** en az 1000 adımdan sonra üç panelde state etkisi <1e−3 ve ek adımlardan topoloji/merkez çizgisi katkısı görülmezse kaydedip durur. Bu bilimsel ret kararı değil, gereksiz GPU harcamasını kesen görünür kuraldır. `stop_on_no_flow_signal=False` aynı matematikle bu bütçe politikasını kapatır.

Yerel checkpoint her 500 adım veya 30 dakika; ayrıca pahalı panel öncesi ve oturum sonunda. `backup_dir` varsa yalnız tamamlanmış checkpoint kopyalanır. 6 saatlik oturum limiti tamamlanmış işlem sonunda kontrol edilir; sürmekte olan validation’ı yarıda kesen katı süre sınırı değildir. Notebook’u kapatmak GPU faturasını durdurmaz; sağlayıcı panelinden GPU’yu sonlandırman gerekir.

In [ ]:
if not CPU_CACHE_ONLY:
    _latest = Path(CONFIG["run_dir"])/"latest.pt"
    RESUME_PATH = str(_latest) if _latest.exists() else None
    # Başka GPU örneğine geçtiysen bunu kalıcı yedeğin latest.pt yolu olarak değiştirebilirsin.
    STATUS = train(MODEL,CONFIG,INFO,SPLITS,resume_path=RESUME_PATH)
    display(STATUS)

## 7. Flow başarılı mı? Sonuç ekranını oku

**Loss düşmesi yeterli değil.** Ana karşılaştırma aynı modelin NFE=4 ve NFE=1 çıktılarıdır. State shuffle etkisi yüksekken sonuç kötüleşebilir; yalnızca state kullanıldığını gösterir. SDF global mesafe tanımına sahip diye ağ global topoloji garantisi kazanmaz. Daha fazla adımın monoton iyileşmesi şart veya garanti değildir.

| Ekran | Aranan sinyal |
|---|---|
| Dice, baseline’a göre | Kalite kaybı marjı: 0.001 mutlak Dice. |
| Filtre sonrası β₀ hatası | Ana topoloji metriği; hem baseline hem NFE=1’den düşük olmalı. |
| clDice, kaçırılan GT merkez çizgisi oranı | Bileşen değişiminin bağlantılılıkla uyumlu olup olmadığını destekler. |
| Ham / filtreli bileşen ayrımı | Küçük adacık temizliğini gerçek bağlantı onarımından ayırmaya yardım eder. |
| Temiz kafa state duyarlılığı | Çok düşükse doğrudan-regresyon rejimine işaret eder; yüksek olması başarı değildir. |

Dökümdeki 2.119 bileşen **staged yöntemin çıktısıdır**, baseline’ın büyük boşlukları olduğunun kanıtı değildir. Bu nedenle gerçek baseline burada aynı protokolle yeniden ölçülür. 5 vaka panelinden yayın/klinik sonuç çıkarılmaz.

İskelet inceltmesi dolu bir maskeyi boş iskelete indirgerse ilgili clDice/gap `null` gösterilir; sıfır uydurulmaz. `skeleton_failure_sides` sütunu bunu bildirir. Bu durumda eksik metrikten otomatik durma/başarı sonucu çıkarılmaz.

In [ ]:
if not CPU_CACHE_ONLY:
    import json
    import matplotlib.pyplot as plt
    _history = Path(CONFIG["run_dir"])/"history.jsonl"
    if _history.exists():
        _h = pd.DataFrame([json.loads(line) for line in _history.read_text().splitlines() if line.strip()])
        display(_h.tail())
        fig,ax=plt.subplots(figsize=(9,3));ax.plot(_h.step,_h.loss)
        ax.set(xlabel="Optimizer attempt",ylabel="SDF / weighted FM loss",title="Öğrenme — NFE faydasının kanıtı değildir")
        ax.grid(alpha=.2);plt.show()
    _probe_files=sorted(Path(CONFIG["run_dir"]).glob("probe_*.json"))
    _table=[]
    for _file in _probe_files:
        _p=json.loads(_file.read_text())
        for _name,_m in _p["summary"].items():
            _table.append(dict(step=_p["step"],method=_name,**_m,
                               state_sensitivity=_p["state_sensitivity"] if _name.startswith("flow") else None))
    if _table:
        _df=pd.DataFrame(_table)
        display(_df[_df.step==_df.step.max()])
        fig,axes=plt.subplots(1,3,figsize=(14,3.4))
        for _name,_group in _df.groupby("method"):
            for _ax,_key in zip(axes,["dice","betti0_error_filtered","cldice"]):
                _ax.plot(_group.step,_group[_key],marker="o",label=_name);_ax.set_title(_key);_ax.set_xlabel("Adım")
        axes[0].legend();fig.tight_layout();plt.show()
    else: print("Henüz panel yok. Loss tek başına flow kararı vermez.")

## 8. Tam validation — eğitim bittiğinde bir kez

Bu hücre **varsayılan kapalıdır**: beş vaka yerine bütün validation üzerinde NFE=1 ve primary NFE çıkarımı yapar, ek eğitim yapmaz. Maliyeti ilk panelin vaka başına süresinden kabaca hesaplayabilirsin. `RUN_FULL_VALIDATION=True` yapıp hücreyi çalıştır.

Hasta bazında eşli bootstrap L/R’ı bağımsız hastalar saymaz. İlan edilen karar: Dice farkının %95 alt sınırı ≥−0.001; filtreli β₀ hata farkının %95 üst sınırı hem baseline’a hem NFE=1’e karşı <0. clDice ve gap ölçümleri destekleyicidir. Tek fold ve training sırasında görülen validation bağımsız test değildir; checkpoint seçimi iyimserlik yaratabilir.

`max_gap_extent_mm` eksik iskelet parçasının kutu köşegenidir; geodezik uzunluk değildir. HD95 varsayılan kapalı, istenirse sadece burada açılır. Preprocessed spacing’deki HD95’in başka native/TTA/aggregate protokolle eşit olması beklenmez. Native affine export bu notebook’ta üretilmez.

In [ ]:
RUN_FULL_VALIDATION = DEMO  # Gerçek koşunun sonunda True yap.
if not CPU_CACHE_ONLY and RUN_FULL_VALIDATION:
    FINAL = evaluate(MODEL,CONFIG,INFO,SPLITS["val"],nfes=[1,CONFIG["primary_nfe"]],
                     step=STATUS["step"],hd95=CONFIG["final_hd95"],tag="full_validation")
    REPORT = final_report(FINAL,CONFIG)
    display(pd.DataFrame({k:v for k,v in REPORT.items() if k!="decision"}).T)
    display(REPORT["decision"])
elif not CPU_CACHE_ONLY:
    print("Tam validation atlandı; henüz nihai başarı kararı yok.")

## 9. Kapatmadan önce

`latest.pt` ve `previous.pt` devam etmek içindir; `history.jsonl`, `probe_*.csv/json`, `final_report.json`, `config.json`, `environment.json` değerlendirme içindir. Remote her-adım yazımı yerine küçük raporları oturum sonunda tek arşivde alabilirsin. Cache yerel geçici diskteyse GPU örneği silindiğinde kaybolabilir; tekrar hazırlama maliyetini düşünerek cache’i de uygun kalıcı diskte tut.

Bir sonraki oturumda orijinal checkpoint’i yükle, aynı cache’i bağla, eğitim hücresindeki resume yolunu seç. GPU/AMP kütüphane farkları bit düzeyinde aynılığı garanti etmez. **Model, optimizer, scaler, RNG ve örnek sayacı birlikte** devam eder. Sadece model ağırlıklarını yüklemek aynı eğitimin devamı değildir.

Yöntemin mekanizması, cebirsel sıfır durumu, bütün sınırlamalar ve kaynaklar açılan `README_TR.md` dosyasında. Gaussian→SDF + CFM literatürde bilinir; bu notebook’u özgünlük kanıtı olarak sunma. Ölçümler çok adım katkısı göstermiyorsa sonuç bunu açıkça söylemelidir.

In [ ]:
if not CPU_CACHE_ONLY:
    _run=Path(CONFIG["run_dir"])
    _report_zip=_run/"small_reports.zip"
    with zipfile.ZipFile(_report_zip,"w",zipfile.ZIP_DEFLATED) as _zip:
        for _p in sorted(_run.rglob("*")):
            if _p.is_file() and _p.suffix in (".json",".jsonl",".csv"):
                _zip.write(_p,_p.relative_to(_run))
    print("Son checkpoint:",_run/"latest.pt")
    print("Küçük rapor arşivi:",_report_zip)
    print("Ayrıntılı yöntem notu:",CODE_ROOT/"README_TR.md")